# SDC-2023 — Collection Track Maps

Maps the **ground-truth trajectory** of every 2023 session used in the multipath pipeline, so the routes driven can be inspected against the multipath findings. Each session's reference track comes from its `ground_truth.csv` (`LatitudeDegrees` / `LongitudeDegrees`).

Sessions are grouped by geographic area (Mountain View, San Jose, and the September routes). Green marker = start, red marker = end.

## 1. Setup

Uses **folium** for interactive Leaflet maps (same library as `00_coordinate_maps.ipynb`). If it is not installed in the kernel, the next cell installs it.

In [1]:
try:
    import folium
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium'])
    import folium

import os
import glob
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../..'))
RAW_DIR  = os.path.join(BASE_DIR, 'data/01_raw/sdc2023/train')
print('folium', folium.__version__)
print('BASE_DIR =', BASE_DIR)

folium 0.20.0
BASE_DIR = c:\Users\Dell\Documents\Warwick\Diss\Code\Dissertation_Trial\GNSS_Multipath_Project


## 2. Which sessions were used?

Re-derives the exact set of sessions that fed the classifier: every 2023 session containing at least one device that reports `MultipathIndicator = 1`.

In [2]:
gnss_files = sorted(glob.glob(os.path.join(RAW_DIR, '2023-*', '*', 'device_gnss.csv')))

used_sessions = set()
for f in gnss_files:
    parts = f.replace('\\', '/').split('/')
    sess = parts[-3]
    mp = pd.read_csv(f, usecols=['MultipathIndicator'])['MultipathIndicator']
    if (mp == 1).any():
        used_sessions.add(sess)

used_sessions = sorted(used_sessions)
print(f'{len(used_sessions)} sessions with multipath:')
for s in used_sessions:
    print('  ', s)

17 sessions with multipath:
   2023-03-08-21-34-us-ca-mtv-u
   2023-05-09-21-32-us-ca-mtv-pe1
   2023-05-16-19-55-us-ca-mtv-xe1
   2023-05-19-20-10-us-ca-mtv-ie2
   2023-05-24-20-26-us-ca-sjc-ge2
   2023-05-25-19-10-us-ca-sjc-be2
   2023-05-25-20-11-us-ca-sjc-he2
   2023-05-26-18-51-us-ca-sjc-ge2
   2023-09-05-20-13-us-ca
   2023-09-05-23-07-us-ca-routen
   2023-09-06-00-01-us-ca-routen
   2023-09-06-18-04-us-ca
   2023-09-06-18-47-us-ca
   2023-09-06-22-49-us-ca-routebb1
   2023-09-07-18-59-us-ca
   2023-09-07-19-33-us-ca
   2023-09-07-22-47-us-ca-routebc2


## 3. Load Ground-Truth Tracks

For each used session, load one representative `ground_truth.csv` (all devices in a session follow the same reference trajectory) and downsample for a light-weight polyline.

In [3]:
def load_track(session):
    gts = sorted(glob.glob(os.path.join(RAW_DIR, session, '*', 'ground_truth.csv')))
    if not gts:
        return None
    g = pd.read_csv(gts[0])
    g = g.dropna(subset=['LatitudeDegrees', 'LongitudeDegrees'])
    g = g[(g['LatitudeDegrees'].between(-90, 90)) & (g['LongitudeDegrees'].between(-180, 180))]
    if g.empty:
        return None
    step = max(1, len(g) // 2000)   # cap ~2000 pts per track
    pts = list(zip(g['LatitudeDegrees'][::step], g['LongitudeDegrees'][::step]))
    return pts

tracks = {}
for s in used_sessions:
    pts = load_track(s)
    if pts:
        tracks[s] = pts
        clat, clon = np.mean([p[0] for p in pts]), np.mean([p[1] for p in pts])
        print(f'{s:38} {len(pts):>5} pts   center=({clat:.4f}, {clon:.4f})')
    else:
        print(f'{s:38} no ground-truth coordinates')

2023-03-08-21-34-us-ca-mtv-u            1102 pts   center=(37.3785, -121.9969)
2023-05-09-21-32-us-ca-mtv-pe1          2132 pts   center=(37.3730, -122.1416)
2023-05-16-19-55-us-ca-mtv-xe1          2323 pts   center=(37.3589, -122.0986)
2023-05-19-20-10-us-ca-mtv-ie2          1230 pts   center=(37.3928, -122.0818)
2023-05-24-20-26-us-ca-sjc-ge2          1386 pts   center=(37.2888, -121.9418)
2023-05-25-19-10-us-ca-sjc-be2          1393 pts   center=(37.2830, -122.0107)
2023-05-25-20-11-us-ca-sjc-he2          1264 pts   center=(37.2490, -121.9438)
2023-05-26-18-51-us-ca-sjc-ge2          1462 pts   center=(37.2891, -121.9403)
2023-09-05-20-13-us-ca                  1704 pts   center=(37.3752, -121.8610)
2023-09-05-23-07-us-ca-routen           1564 pts   center=(37.3651, -122.0428)
2023-09-06-00-01-us-ca-routen           1651 pts   center=(37.3655, -122.0442)
2023-09-06-18-04-us-ca                  1452 pts   center=(37.2273, -121.7530)
2023-09-06-18-47-us-ca                  1485 pts   c

## 4. Overview Map — All Sessions

Every used session on one interactive map, colour-cycled, with a satellite-imagery layer available from the layer control (top-right).

In [4]:
COLORS = ['blue', 'red', 'green', 'orange', 'purple', 'darkblue', 'darkred',
          'cadetblue', 'darkgreen', 'black', 'pink', 'darkpurple', 'lightblue',
          'beige', 'lightgreen', 'gray', 'lightred']

def build_map(session_pts, zoom=11):
    all_pts = [p for pts in session_pts.values() for p in pts]
    center = [np.mean([p[0] for p in all_pts]), np.mean([p[1] for p in all_pts])]
    m = folium.Map(location=center, zoom_start=zoom, tiles='cartodbpositron')
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Esri Satellite', overlay=False, control=True).add_to(m)
    for i, (label, pts) in enumerate(session_pts.items()):
        color = COLORS[i % len(COLORS)]
        folium.PolyLine(pts, color=color, weight=4, opacity=0.8, tooltip=label).add_to(m)
        folium.Marker(pts[0],  popup=f'Start: {label}', icon=folium.Icon(color='green', icon='play')).add_to(m)
        folium.Marker(pts[-1], popup=f'End: {label}',   icon=folium.Icon(color=color,  icon='stop')).add_to(m)
    folium.LayerControl().add_to(m)
    return m

overview = build_map(tracks, zoom=9)
display(overview)

## 5. Regional Maps

The overview spans the whole Bay Area, so each geographic cluster is also shown zoomed-in. Sessions are grouped by the location tag in their folder name.

In [5]:
def region_of(session):
    if 'mtv' in session: return 'Mountain View'
    if 'sjc' in session: return 'San Jose'
    return 'September routes (SF / hwy)'

regions = {}
for s, pts in tracks.items():
    regions.setdefault(region_of(s), {})[s] = pts

for region, sess_pts in regions.items():
    print(f'\n### {region}  —  {len(sess_pts)} session(s)')
    display(build_map(sess_pts, zoom=12))


### Mountain View  —  4 session(s)



### San Jose  —  4 session(s)



### September routes (SF / hwy)  —  9 session(s)


## 6. Session Summary Table

Start/end coordinates and point counts for every mapped session.

In [6]:
rows = []
for s, pts in tracks.items():
    rows.append({'session': s, 'region': region_of(s), 'n_points': len(pts),
                 'start_lat': round(pts[0][0], 5), 'start_lon': round(pts[0][1], 5),
                 'end_lat': round(pts[-1][0], 5), 'end_lon': round(pts[-1][1], 5)})
summary = pd.DataFrame(rows).sort_values(['region', 'session']).reset_index(drop=True)
summary

,session,region,n_points,start_lat,start_lon,end_lat,end_lon
0,2023-03-08-21-34-us-ca-mtv-u,Mountain View,1102,37.38298,-122.03846,37.38294,-122.03839
1,2023-05-09-21-32-us-ca-mtv-pe1,Mountain View,2132,37.33003,-122.08557,37.33003,-122.08557
2,2023-05-16-19-55-us-ca-mtv-xe1,Mountain View,2323,37.32215,-121.99492,37.32214,-121.99492
3,2023-05-19-20-10-us-ca-mtv-ie2,Mountain View,1230,37.40264,-122.07880,37.40263,-122.07883
4,2023-05-24-20-26-us-ca-sjc-ge2,San Jose,1386,37.29193,-121.93141,37.29194,-121.93141
5,2023-05-25-19-10-us-ca-sjc-be2,San Jose,1393,37.28934,-121.98942,37.28932,-121.98940
6,2023-05-25-20-11-us-ca-sjc-he2,San Jose,1264,37.23511,-121.96160,37.23512,-121.96161
7,2023-05-26-18-51-us-ca-sjc-ge2,San Jose,1462,37.29202,-121.93121,37.29202,-121.93121
8,2023-09-05-20-13-us-ca,September routes (SF / hwy),1704,37.36600,-121.84982,37.36607,-121.84989
9,2023-09-05-23-07-us-ca-routen,September routes (SF / hwy),1564,37.36516,-122.03166,37.36516,-122.03166
